In [ ]:
import pandas as pd
import numpy as np

<h1>1. EXTRACT</h1>

<h3>1.1. Competition Extract</h3>

In [ ]:
# xử lý dữ liệu về các giải đấu trong nước anh
competitions = pd.read_csv("./FbData/competitions.csv")
# HIGHLIGHT
uk_competitions = competitions[competitions['competition_id'] == 'GB1']
print(uk_competitions)

   competition_id competition_code            name    sub_type  \
29            GB1   premier-league  premier-league  first_tier   

               type  country_id country_name domestic_league_code  \
29  domestic_league         189      England                  GB1   

   confederation  total_clubs  \
29        europa         20.0   

                                                  url  
29  https://www.transfermarkt.co.uk/premier-league...  


<h3>1.2. Club Extract</h3>

In [ ]:
# xử lý dữ liệu về câu lạc bộ tham gia các giải đấu trong nước anh
clubs = pd.read_csv("./FbData/clubs.csv")
# HIGHLIGHT
uk_clubs = clubs[clubs['domestic_competition_id'] == 'GB1']
print(clubs.shape)

# counts = uk_clubs['club_id'].value_counts()
# duplicated_teams = counts[counts > 1].index.tolist()
# print(duplicated_teams)
print(uk_clubs.shape)

(796, 17)
(37, 17)


<h3>1.3. Game Information Extract</h3>

In [ ]:
# xử lý thông tin chi tiết về trận đấu
games = pd.read_csv("./FbData/games.csv")
# HIGHLIGHT
uk_games = games[games['competition_id'] == 'GB1']
print(uk_games.shape)


(5249, 23)


In [ ]:
# xử lý dữ liệu về các trận đấu của các câu lạc bộ đó.
club_games_info = pd.read_csv("./FbData/club_games.csv")
# HIGHLIGHT
uk_club_games = club_games_info[club_games_info['game_id'].isin(uk_games['game_id'])]
print(uk_club_games.columns)

Index(['game_id', 'club_id', 'own_goals', 'own_position', 'own_manager_name',
       'opponent_id', 'opponent_goals', 'opponent_position',
       'opponent_manager_name', 'hosting', 'is_win'],
      dtype='object')


In [ ]:
game_events = pd.read_csv("./FbData/game_events.csv")
uk_game_events = game_events[game_events['game_id'].isin(uk_games['game_id'])]
print(uk_game_events.columns)

Index(['game_event_id', 'date', 'game_id', 'minute', 'type', 'club_id',
       'club_name', 'player_id', 'description', 'player_in_id',
       'player_assist_id'],
      dtype='object')


<h3>1.4. Player Information Extract</h3>

In [ ]:
# xử lý thông tin ra sân của các cầu thủ
appearances = pd.read_csv("./FbData/appearances.csv")
# HIGHLIGHT
uk_appearances = appearances[appearances['competition_id'] == 'GB1']
print(uk_appearances.columns)

Index(['appearance_id', 'game_id', 'player_id', 'player_club_id',
       'player_current_club_id', 'date', 'player_name', 'competition_id',
       'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played'],
      dtype='object')


In [ ]:
# xử lý các thông tin liên quan đến cầu thủ
# thông tin cá nhân
players = pd.read_csv("./FbData/players.csv")
print(players.shape)

# chỉ lấy thông tin các cầu thủ thi đấu ở các giải của england
# HIGHLIGHT
uk_players = players[players['player_id'].isin(uk_appearances['player_id'].unique())]
print(uk_players.columns)

# giá trị chuyển nhượng
players_value = pd.read_csv("./FbData/player_valuations.csv")
print(players_value.shape)

# HIGHLIGHT
uk_players_value = players_value[players_value['player_id'].isin(uk_players['player_id'])]
print(uk_players_value.columns)

# các cuộc chuyển nhượng
transfers = pd.read_csv("./FbData/transfers.csv")
print(transfers.shape)

# HIGHLIGHT
uk_players_transfer = transfers[transfers['player_id'].isin(uk_players['player_id'])]
print(uk_players_transfer.columns)

(47702, 26)
Index(['player_id', 'first_name', 'last_name', 'name', 'last_season',
       'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth',
       'country_of_citizenship', 'date_of_birth', 'sub_position', 'position',
       'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name',
       'image_url', 'international_caps', 'international_goals',
       'current_national_team_id', 'url',
       'current_club_domestic_competition_id', 'current_club_name',
       'market_value_in_eur', 'highest_market_value_in_eur'],
      dtype='object')
(616377, 6)
Index(['player_id', 'date', 'market_value_in_eur', 'current_club_name',
       'current_club_id', 'player_club_domestic_competition_id'],
      dtype='object')
(157186, 10)
Index(['player_id', 'transfer_date', 'transfer_season', 'from_club_id',
       'to_club_id', 'from_club_name', 'to_club_name', 'transfer_fee',
       'market_value_in_eur', 'player_name'],
      dtype='object')


<h1>2. Transform Data</h1>

<h3>2.1. Dim_Competition</h3>

In [ ]:
# Dữ liệu cho competition
dim_competion = uk_competitions[[
    'competition_id', 'competition_code', 'name', 'sub_type', 'type',
    'domestic_league_code',  'confederation', 'total_clubs'
    ]]
#xử lý total_clubs bị null trong giải có id là CGB
dim_competion.loc[dim_competion['competition_id'] == 'CGB', 'total_clubs'] = 92
dim_competion['competition_sk'] = range(1, len(dim_competion)+1)
# HIGHLIGHT
print(dim_competion)

   competition_id competition_code            name    sub_type  \
29            GB1   premier-league  premier-league  first_tier   

               type domestic_league_code confederation  total_clubs  \
29  domestic_league                  GB1        europa         20.0   

    competition_sk  
29               1  


C:\Users\7610\AppData\Local\Temp\ipykernel_3440\3024085669.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dim_competion['competition_sk'] = range(1, len(dim_competion)+1)


<h3>2.2. Dim_Club</h3>

In [ ]:
# xử lý dữ liệu cho dim_club
# club_ids = uk_appearances['player_current_club_id'].unique() # 423 
# new_clubs = clubs[clubs['club_id'].isin(club_ids)]
# print(new_clubs.shape)
dim_club = clubs[[
    'club_id', 'club_code', 'name', 'domestic_competition_id', 'squad_size', 'average_age',
    'foreigners_number', 'foreigners_percentage', 'stadium_name',
       'stadium_seats', 'net_transfer_record'
]].copy()
def parse_net(value):
    if pd.isna(value):
        return None
    # loại bỏ ký hiệu € và khoảng trắng
    v = value.replace('€', '').replace('+', '').strip().lower()
    multiplier = 1
    if v.endswith('m'):
        multiplier = 1_000_000
        v = v[:-1]
    elif v.endswith('k'):
        multiplier = 1_000
        v = v[:-1]
    try:
        return float(v) * multiplier
    except ValueError:
        return None
dim_club['foreigners_percentage'] = dim_club['foreigners_percentage'].fillna(0)
num_cols = [
    'squad_size', 'average_age',
]

for col in num_cols:
    dim_club[col] = dim_club[col].fillna(dim_club[col].median())
dim_club.loc[:, 'net_transfer_record'] = dim_club['net_transfer_record'].apply(parse_net)
dim_club = dim_club.copy()
dim_club['club_sk'] = range(1, len(dim_club)+1)
# HIGHLIGHT
print(dim_club.isna().sum())


club_id                    0
club_code                  0
name                       0
domestic_competition_id    0
squad_size                 0
average_age                0
foreigners_number          0
foreigners_percentage      0
stadium_name               0
stadium_seats              0
net_transfer_record        0
club_sk                    0
dtype: int64


<h3>2.3. Dim_Player</h3>

In [ ]:
# xử lý dữ liệu cho dim_player

dim_player = uk_players[[
    'player_id', 'first_name', 'last_name', 'name', 'date_of_birth', 'country_of_birth', 
    'city_of_birth', 'country_of_citizenship', 'position', 'sub_position', 'foot', 'height_in_cm',
    'image_url', 'market_value_in_eur', 'highest_market_value_in_eur'
]]

# chỉ lấy ngày 
dim_player = dim_player.copy()
dim_player['date_of_birth'] = pd.to_datetime(dim_player['date_of_birth'])

# xử lý nan ở foot 
choices = ['left', 'right', 'both']
mask = dim_player['foot'].isna()
dim_player.loc[mask, 'foot'] = np.random.choice(choices, size=mask.sum())

# xử lý nan ở height in cm
mask = dim_player['height_in_cm'].isna()
dim_player.loc[mask, 'height_in_cm'] = np.random.randint(
    175, 200,
    size=mask.sum()
)

# xử lý ngày sinh bị nan
mask = dim_player['date_of_birth'].isna()
current_year = 2026
ages = np.random.randint(18, 41, size=mask.sum())
dim_player.loc[mask, 'date_of_birth'] = pd.to_datetime(
    current_year - ages, format='%Y'
) + pd.to_timedelta(np.random.randint(0, 365, size=mask.sum()), unit='D')

# xử lý giá trị nan ở giá trị chuyển nhượng
mask = dim_player['market_value_in_eur'].isna()
    # log-normal: mean ~ log(5 triệu), sigma điều chỉnh độ phân tán
values = np.random.lognormal(mean=np.log(5_000_000), sigma=1.0, size=mask.sum())
    # giới hạn khoảng hợp lý (ví dụ 100k → 150 triệu)
values = np.clip(values, 100_000, 150_000_000)
dim_player.loc[mask, 'market_value_in_eur'] = values.round()

# xử lý giá trị nan ở highest_market_value_in_eur
mask = dim_player['highest_market_value_in_eur'].isna()
values = np.random.lognormal(mean=np.log(5_000_000), sigma=1.0, size=mask.sum())
values = np.clip(values, 100_000, 150_000_000)
dim_player.loc[mask, 'highest_market_value_in_eur'] = values.round()

# xử lý player_sk
dim_player['player_sk'] = range(1, len(dim_player)+1)

# HIGHLIGHT
print(dim_player.shape)

(2441, 16)


<h3>2.4. Dim_Date</h3>

In [ ]:
# xử lý dữ liệu cho dim_date
dim_date = pd.DataFrame({
    'date': pd.date_range(start='2010-01-01', end='2030-12-31')
})

dim_date['date_sk'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['day'] = dim_date['date'].dt.day
dim_date['month'] = dim_date['date'].dt.month
dim_date['year'] = dim_date['date'].dt.year
dim_date['weekday'] = dim_date['date'].dt.day_name()

def get_season(date):
    if date.month >= 7:
        return f"{date.year}/{date.year+1}"
    else:
        return f"{date.year-1}/{date.year}"

dim_date['season'] = dim_date['date'].apply(get_season)

# reorder đúng schema
dim_date = dim_date[
    ['date_sk', 'date', 'day', 'month', 'year', 'weekday', 'season']
]
# HIGHLIGHT
print(dim_date)

       date_sk       date  day  month  year   weekday     season
0     20100101 2010-01-01    1      1  2010    Friday  2009/2010
1     20100102 2010-01-02    2      1  2010  Saturday  2009/2010
2     20100103 2010-01-03    3      1  2010    Sunday  2009/2010
3     20100104 2010-01-04    4      1  2010    Monday  2009/2010
4     20100105 2010-01-05    5      1  2010   Tuesday  2009/2010
...        ...        ...  ...    ...   ...       ...        ...
7665  20301227 2030-12-27   27     12  2030    Friday  2030/2031
7666  20301228 2030-12-28   28     12  2030  Saturday  2030/2031
7667  20301229 2030-12-29   29     12  2030    Sunday  2030/2031
7668  20301230 2030-12-30   30     12  2030    Monday  2030/2031
7669  20301231 2030-12-31   31     12  2030   Tuesday  2030/2031

[7670 rows x 7 columns]


<h3>2.5. Fact_Game</h3>

In [ ]:
# xử lý dữ liệu cho fact_game
# print(uk_games.columns)
fact_game = uk_games[[
    'game_id', 'competition_id', 'season', 'round', 'home_club_goals', 'away_club_goals', 'home_club_id', 'away_club_id',
       'home_club_position', 'away_club_position', 'home_club_manager_name',
       'away_club_manager_name', 'stadium', 'attendance', 'referee', 'url',
       'home_club_formation', 'away_club_formation', 'date'
]]
fact_game = fact_game.rename(columns={
    'home_club_goals' : 'home_goals',
    'away_club_goals' : 'away_goals',
    'home_club_manager_name': 'home_manager',
    'away_club_manager_name': 'away_manager',
    'home_club_formation': 'home_formation',
    'away_club_formation': 'away_formation'
})

missing_home = set(fact_game['home_club_id']) - set(dim_club['club_id'])
missing_away = set(fact_game['away_club_id']) - set(dim_club['club_id'])

print("Home clubs missing in dim_club:", missing_home)
print("Away clubs missing in dim_club:", missing_away)
same_club_in_game = fact_game[fact_game['home_club_id'] == fact_game['away_club_id']]
print(same_club_in_game[['home_club_id', 'away_club_id', 'season', 'round']])
# đảm bảo cùng dtype
fact_game['home_club_id'] = fact_game['home_club_id'].astype(str)
fact_game['away_club_id'] = fact_game['away_club_id'].astype(str)
dim_club['club_id'] = dim_club['club_id'].astype(str)

# map home_club_sk
fact_game = fact_game.merge(
    dim_club[['club_id', 'club_sk']],
    left_on='home_club_id',
    right_on='club_id',
    how='left'
).rename(columns={'club_sk': 'home_club_sk'}).drop(columns=['club_id'])

# map away_club_sk
fact_game = fact_game.merge(
    dim_club[['club_id', 'club_sk']],
    left_on='away_club_id',
    right_on='club_id',
    how='left'
).rename(columns={'club_sk': 'away_club_sk'}).drop(columns=['club_id', 'home_club_id', 'away_club_id'])

# map competition_sk
fact_game = fact_game.merge(
    dim_competion[['competition_id', 'competition_sk']],
    left_on='competition_id',
    right_on='competition_id',
    how='left'
).drop(columns=['competition_id'])

#map date_sk
fact_game['date'] = pd.to_datetime(fact_game['date'])
dim_date['date'] = pd.to_datetime(dim_date['date'])
fact_game = fact_game.merge(
    dim_date[['date', 'date_sk']],
    left_on='date',
    right_on='date',
    how='left'
).drop(columns=['date', 'home_formation', 'away_formation', 'home_club_position', 'away_club_position'])
fact_game['game_sk'] = range(1, len(fact_game)+1)

# xử lý dữ liệ bị null ở attendance 
# fill NaN bằng median
fact_game['attendance'] = fact_game['attendance'].fillna(fact_game['attendance'].median())
# HIGHLIGHT
# print(fact_game.isna().sum())

Home clubs missing in dim_club: set()
Away clubs missing in dim_club: set()
Empty DataFrame
Columns: [home_club_id, away_club_id, season, round]
Index: []


<h3>2.6. Fact_Game_Event</h3>

In [ ]:
# xử lý dữ liệu game_events
fact_game_events = uk_game_events.rename(columns={
    'type' : 'event_type'
})

# map player_in_id -> player_in_sk
fact_game_events = fact_game_events.merge(
    dim_player[['player_id', 'player_sk']],
    left_on = 'player_in_id',
    right_on='player_id',
    how='left'
).rename(columns={'player_sk': 'player_in_sk'}).drop(columns=['player_in_id', 'player_id_x', 'player_id_y'])

# map player_assist_id -> player_assist_sk
fact_game_events = fact_game_events.merge(
    dim_player[['player_id', 'player_sk']],
    left_on='player_assist_id',
    right_on='player_id',
    how='left'
).rename(columns={'player_sk' : 'player_assist_sk'}).drop(columns=['player_assist_id'])

# map club_id -> club_sk 
fact_game_events['club_id'] = fact_game_events['club_id'].astype(int)
dim_club['club_id'] = dim_club['club_id'].astype(int)
fact_game_events = fact_game_events.merge(
    dim_club[['club_id', 'club_sk']],
    left_on='club_id',
    right_on='club_id',
    how='left'
).drop(columns=['club_id'])

# map player_id -> player_sk
fact_game_events = fact_game_events.merge(
    dim_player[['player_id', 'player_sk']],
    left_on='player_id',
    right_on='player_id',
    how='left'
).drop(columns=['player_id'])

# map date_sk
fact_game_events['date'] = pd.to_datetime(fact_game_events['date'])
fact_game_events = fact_game_events.merge(
    dim_date[['date', 'date_sk']],
    left_on='date',
    right_on='date',
    how='left'
).drop(columns=['date'])

#map game_id -> game_sk
fact_game_events = fact_game_events.merge(
    fact_game[['game_id', 'game_sk']],
    left_on='game_id',
    right_on='game_id',
    how='left'
).drop(columns=['game_id'])

# tạo game_event_sk
fact_game_events['game_event_sk'] = range(1, len(fact_game_events) + 1)
# chuyển tất cả NaN thành None
fact_game_events = fact_game_events.where(pd.notna(fact_game_events), None)
# HIGHLIGHT
print(fact_game_events.isna().sum())

game_event_id           0
minute                  0
event_type              0
club_name               0
description          6789
player_in_sk        33771
player_assist_sk    55505
club_sk                 0
player_sk           55505
date_sk                 0
game_sk                 0
game_event_sk           0
dtype: int64


<h3>2.7. Fact_Club_Game_Stats</h3>

In [ ]:
fact_club_game_stats = uk_club_games.copy()
print(fact_club_game_stats.isna().sum())

# map game_id -> game_sk
fact_club_game_stats = fact_club_game_stats.merge(
    fact_game[['game_id', 'game_sk']],
    left_on = 'game_id',
    right_on = 'game_id',
    how='left'
).drop(columns=['game_id'])

# map club_id -> club_sk
fact_club_game_stats = fact_club_game_stats.merge(
    dim_club[['club_id', 'club_sk']],
    left_on='club_id',
    right_on='club_id',
    how='left'
).drop(columns=['club_id'])

# map opponent_id -> opponent_club_sk
a_dim_club = dim_club.copy().rename(columns={'club_sk' : 'opponent_club_sk'})
fact_club_game_stats = fact_club_game_stats.merge(
    a_dim_club[['club_id', 'opponent_club_sk']],
    left_on='opponent_id',
    right_on='club_id',
    how='left'
).drop(columns=['opponent_id', 'club_id', 'own_position', 'opponent_position'])

# tạo fact_club_game_stats_sk 
fact_club_game_stats['club_game_sk'] = range(1, len(fact_club_game_stats) + 1)

# HIGHLIGHT
print(fact_club_game_stats.isna().sum())

game_id                  0
club_id                  0
own_goals                0
own_position             0
own_manager_name         0
opponent_id              0
opponent_goals           0
opponent_position        0
opponent_manager_name    0
hosting                  0
is_win                   0
dtype: int64
own_goals                0
own_manager_name         0
opponent_goals           0
opponent_manager_name    0
hosting                  0
is_win                   0
game_sk                  0
club_sk                  0
opponent_club_sk         0
club_game_sk             0
dtype: int64


<h3>2.8. Fact_Player_Game_Stats</h3>

In [ ]:
fact_player_game_stats = uk_appearances.copy()
def map_to_sk(left, right, sr, df, source):
    cols_to_drop = [left] if left == right else [left, right]
    
    return df.merge(
        source[[right, sr]],
        left_on=left,
        right_on=right,
        how='left'
    ).drop(columns=cols_to_drop)

# map game_id -> game_sk
fact_player_game_stats = map_to_sk('game_id', 'game_id', 'game_sk', df=fact_player_game_stats,source=fact_game)

# map player_id -> player_sk
fact_player_game_stats = map_to_sk('player_id', 'player_id', 'player_sk', df=fact_player_game_stats, source=dim_player)

# map player_club_id -> club_sk
fact_player_game_stats = map_to_sk('player_club_id', 'club_id', 'club_sk', df=fact_player_game_stats, source=dim_club)

# map player_current_club_id -> club_current_sk
new_dim_club = dim_club.copy()
new_dim_club = new_dim_club.rename(columns={
    'club_sk': 'club_current_sk'
})
fact_player_game_stats = map_to_sk(
    'player_current_club_id', 'club_id', 'club_current_sk',
    df=fact_player_game_stats,
    source=new_dim_club
)

# map competition_id -> competition_sk
fact_player_game_stats = map_to_sk(
    left='competition_id', right='competition_id',
    sr='competition_sk',
    df=fact_player_game_stats,
    source=dim_competion
)

# map date -> date_sk
fact_player_game_stats['date'] = pd.to_datetime(fact_player_game_stats['date'])
fact_player_game_stats = map_to_sk(
    left='date', right='date',
    sr='date_sk', 
    df=fact_player_game_stats,
    source=dim_date
)
fact_player_game_stats = fact_player_game_stats.drop(columns=['player_name'])
fact_player_game_stats['appearance_sk'] = range(1, len(fact_player_game_stats) + 1)
# HIGHLIGHT
print(fact_player_game_stats.isna().sum())


appearance_id         0
yellow_cards          0
red_cards             0
goals                 0
assists               0
minutes_played        0
game_sk               0
player_sk             0
club_sk               0
club_current_sk    2575
competition_sk        0
date_sk               0
appearance_sk         0
dtype: int64


<h3>2.9. Fact_Player_Market_Value</h3>

In [ ]:
fact_player_market_value = uk_players_value.copy().drop(columns=['current_club_name', 'player_club_domestic_competition_id'])
# map player_id -> player_sk
fact_player_market_value = map_to_sk(
    left='player_id', right='player_id',
    sr='player_sk',
    df=fact_player_market_value,
    source=dim_player
)
# map date -> date_sk
fact_player_market_value['date'] = pd.to_datetime(fact_player_market_value['date'])
fact_player_market_value = map_to_sk(
    left='date', right='date',
    sr='date_sk',
    df=fact_player_market_value,
    source=dim_date
)
#map current_club_id -> club_sk
fact_player_market_value = map_to_sk(
    left='current_club_id',
    right='club_id',
    sr='club_sk',
    df=fact_player_market_value,
    source=dim_club
)
# tạo sk cho fact_player_market_value
fact_player_market_value['market_value_sk'] = range(1, len(fact_player_market_value) + 1)
# HIGHLIGHT
print(fact_player_market_value.isna().sum())


market_value_in_eur       0
player_sk                 0
date_sk                4642
club_sk                3303
market_value_sk           0
dtype: int64


<h3>2.10. Fact_Player_Transfer</h3>

In [ ]:
fact_player_transfer = uk_players_transfer.copy().drop(columns=['from_club_name', 'to_club_name']).rename(columns={'transfer_season' : 'season'})
# map player_id -> player_sk
fact_player_transfer = map_to_sk(
    left='player_id',
    right='player_id',
    sr='player_sk',
    df=fact_player_transfer,
    source=dim_player
)

# map transfer_date -> transfer_date_sk
fact_player_transfer['transfer_date'] = pd.to_datetime(fact_player_transfer['transfer_date'])
fact_player_transfer = map_to_sk(
    left='transfer_date',
    right='date',
    sr='date_sk',
    df=fact_player_transfer,
    source=dim_date
).rename(columns={'date_sk' : 'transfer_date_sk'})

# map from_club_id -> from_club_sk
fact_player_transfer = map_to_sk(
    left='from_club_id',
    right='club_id',
    sr='club_sk',
    df=fact_player_transfer,
    source=dim_club
).rename(columns={'club_sk' : 'from_club_sk'})

# map to_club_id -> to_club_sk
fact_player_transfer = map_to_sk(
    left='to_club_id',
    right='club_id',
    sr='club_sk',
    df=fact_player_transfer,
    source=dim_club
).rename(columns={'club_sk' : 'to_club_sk'})

fact_player_transfer['transfer_sk'] = range(1, len(fact_player_transfer) + 1)
# HIGHLIGHT
print(fact_player_transfer.columns)

Index(['season', 'transfer_fee', 'market_value_in_eur', 'player_name',
       'player_sk', 'transfer_date_sk', 'from_club_sk', 'to_club_sk',
       'transfer_sk'],
      dtype='object')


<h1>3. VALIDATION - FK Mapping</h1>

In [ ]:
# Kiểm tra primary key uniqueness ở dimension tables
print("\n✓ DIM_COMPETITION:")
print(f"  Total rows: {len(dim_competion)}")
print(f"  Unique competition_sk: {dim_competion['competition_sk'].nunique()}")
print(f"  Has NULL in competition_sk: {dim_competion['competition_sk'].isna().sum()}")

print("\n✓ DIM_CLUB:")
print(f"  Total rows: {len(dim_club)}")
print(f"  Unique club_sk: {dim_club['club_sk'].nunique()}")
print(f"  Has NULL in club_sk: {dim_club['club_sk'].isna().sum()}")

print("\n✓ DIM_PLAYER:")
print(f"  Total rows: {len(dim_player)}")
print(f"  Unique player_sk: {dim_player['player_sk'].nunique()}")
print(f"  Has NULL in player_sk: {dim_player['player_sk'].isna().sum()}")

print("\n✓ DIM_DATE:")
print(f"  Total rows: {len(dim_date)}")
print(f"  Unique date_sk: {dim_date['date_sk'].nunique()}")
print(f"  Has NULL in date_sk: {dim_date['date_sk'].isna().sum()}")

3.1. PRIMARY KEY UNIQUENESS CHECK

✓ DIM_COMPETITION:
  Total rows: 1
  Unique competition_sk: 1
  Has NULL in competition_sk: 0

✓ DIM_CLUB:
  Total rows: 796
  Unique club_sk: 796
  Has NULL in club_sk: 0

✓ DIM_PLAYER:
  Total rows: 2441
  Unique player_sk: 2441
  Has NULL in player_sk: 0

✓ DIM_DATE:
  Total rows: 7670
  Unique date_sk: 7670
  Has NULL in date_sk: 0


In [ ]:
print("\n" + "="*70)
print("3.2. FACT TABLE - FOREIGN KEY INTEGRITY CHECK")
print("="*70)

# FACT_GAME
print("\n✓ FACT_GAME:")
print(f"  Total rows: {len(fact_game)}")
print(f"  Unique game_sk: {fact_game['game_sk'].nunique()}")
print(f"  NULL in game_sk: {fact_game['game_sk'].isna().sum()}")
print(f"  NULL in home_club_sk: {fact_game['home_club_sk'].isna().sum()}")
print(f"  NULL in away_club_sk: {fact_game['away_club_sk'].isna().sum()}")
print(f"  NULL in competition_sk: {fact_game['competition_sk'].isna().sum()}")
print(f"  NULL in date_sk: {fact_game['date_sk'].isna().sum()}")

# Verify FK referencing
home_fk_valid = fact_game['home_club_sk'].isin(dim_club['club_sk']).sum()
away_fk_valid = fact_game['away_club_sk'].isin(dim_club['club_sk']).sum()
comp_fk_valid = fact_game['competition_sk'].isin(dim_competion['competition_sk']).sum()
date_fk_valid = fact_game['date_sk'].isin(dim_date['date_sk']).sum()

print(f"  home_club_sk valid refs: {home_fk_valid}/{len(fact_game)} = {home_fk_valid/len(fact_game)*100:.2f}%")
print(f"  away_club_sk valid refs: {away_fk_valid}/{len(fact_game)} = {away_fk_valid/len(fact_game)*100:.2f}%")
print(f"  competition_sk valid refs: {comp_fk_valid}/{len(fact_game)} = {comp_fk_valid/len(fact_game)*100:.2f}%")
print(f"  date_sk valid refs: {date_fk_valid}/{len(fact_game)} = {date_fk_valid/len(fact_game)*100:.2f}%")

# FACT_GAME_EVENTS
print("\n✓ FACT_GAME_EVENTS:")
print(f"  Total rows: {len(fact_game_events)}")
print(f"  NULL in game_event_sk: {fact_game_events['game_event_sk'].isna().sum()}")
print(f"  NULL in game_sk: {fact_game_events['game_sk'].isna().sum()}")
print(f"  NULL in player_in_sk: {fact_game_events['player_in_sk'].isna().sum()}")
print(f"  NULL in player_assist_sk: {fact_game_events['player_assist_sk'].isna().sum()}")
print(f"  NULL in club_sk: {fact_game_events['club_sk'].isna().sum()}")
print(f"  NULL in player_sk: {fact_game_events['player_sk'].isna().sum()}")
print(f"  NULL in date_sk: {fact_game_events['date_sk'].isna().sum()}")

# Verify FK referencing
game_event_game_valid = fact_game_events['game_sk'].isin(fact_game['game_sk']).sum()
game_event_player_valid = fact_game_events['player_sk'].isin(dim_player['player_sk']).sum()
game_event_club_valid = fact_game_events['club_sk'].isin(dim_club['club_sk']).sum()
game_event_date_valid = fact_game_events['date_sk'].isin(dim_date['date_sk']).sum()

print(f"  game_sk valid refs: {game_event_game_valid}/{len(fact_game_events)} = {game_event_game_valid/len(fact_game_events)*100:.2f}%")
print(f"  player_sk valid refs: {game_event_player_valid}/{len(fact_game_events)} = {game_event_player_valid/len(fact_game_events)*100:.2f}%")
print(f"  club_sk valid refs: {game_event_club_valid}/{len(fact_game_events)} = {game_event_club_valid/len(fact_game_events)*100:.2f}%")
print(f"  date_sk valid refs: {game_event_date_valid}/{len(fact_game_events)} = {game_event_date_valid/len(fact_game_events)*100:.2f}%")


3.2. FACT TABLE - FOREIGN KEY INTEGRITY CHECK

✓ FACT_GAME:
  Total rows: 5249
  Unique game_sk: 5249
  NULL in game_sk: 0
  NULL in home_club_sk: 0
  NULL in away_club_sk: 0
  NULL in competition_sk: 0
  NULL in date_sk: 0
  home_club_sk valid refs: 5249/5249 = 100.00%
  away_club_sk valid refs: 5249/5249 = 100.00%
  competition_sk valid refs: 5249/5249 = 100.00%
  date_sk valid refs: 5249/5249 = 100.00%

✓ FACT_GAME_EVENTS:
  Total rows: 66482
  NULL in game_event_sk: 0
  NULL in game_sk: 0
  NULL in player_in_sk: 33771
  NULL in player_assist_sk: 55505
  NULL in club_sk: 0
  NULL in player_sk: 55505
  NULL in date_sk: 0
  game_sk valid refs: 66482/66482 = 100.00%
  player_sk valid refs: 10977/66482 = 16.51%
  club_sk valid refs: 66482/66482 = 100.00%
  date_sk valid refs: 66482/66482 = 100.00%


In [ ]:
# FACT_CLUB_GAME_STATS
print("\n✓ FACT_CLUB_GAME_STATS:")
print(f"  Total rows: {len(fact_club_game_stats)}")
print(f"  NULL in club_game_sk: {fact_club_game_stats['club_game_sk'].isna().sum()}")
print(f"  NULL in game_sk: {fact_club_game_stats['game_sk'].isna().sum()}")
print(f"  NULL in club_sk: {fact_club_game_stats['club_sk'].isna().sum()}")
print(f"  NULL in opponent_club_sk: {fact_club_game_stats['opponent_club_sk'].isna().sum()}")

game_club_game_valid = fact_club_game_stats['game_sk'].isin(fact_game['game_sk']).sum()
game_club_club_valid = fact_club_game_stats['club_sk'].isin(dim_club['club_sk']).sum()
game_club_opponent_valid = fact_club_game_stats['opponent_club_sk'].isin(dim_club['club_sk']).sum()

print(f"  game_sk valid refs: {game_club_game_valid}/{len(fact_club_game_stats)} = {game_club_game_valid/len(fact_club_game_stats)*100:.2f}%")
print(f"  club_sk valid refs: {game_club_club_valid}/{len(fact_club_game_stats)} = {game_club_club_valid/len(fact_club_game_stats)*100:.2f}%")
print(f"  opponent_club_sk valid refs: {game_club_opponent_valid}/{len(fact_club_game_stats)} = {game_club_opponent_valid/len(fact_club_game_stats)*100:.2f}%")

# FACT_PLAYER_GAME_STATS
print("\n✓ FACT_PLAYER_GAME_STATS (Appearances):")
print(f"  Total rows: {len(fact_player_game_stats)}")
print(f"  NULL in appearance_sk: {fact_player_game_stats['appearance_sk'].isna().sum()}")
print(f"  NULL in game_sk: {fact_player_game_stats['game_sk'].isna().sum()}")
print(f"  NULL in player_sk: {fact_player_game_stats['player_sk'].isna().sum()}")
print(f"  NULL in club_sk: {fact_player_game_stats['club_sk'].isna().sum()}")
print(f"  NULL in club_current_sk: {fact_player_game_stats['club_current_sk'].isna().sum()}")
print(f"  NULL in competition_sk: {fact_player_game_stats['competition_sk'].isna().sum()}")
print(f"  NULL in date_sk: {fact_player_game_stats['date_sk'].isna().sum()}")

app_game_valid = fact_player_game_stats['game_sk'].isin(fact_game['game_sk']).sum()
app_player_valid = fact_player_game_stats['player_sk'].isin(dim_player['player_sk']).sum()
app_club_valid = fact_player_game_stats['club_sk'].isin(dim_club['club_sk']).sum()
app_club_current_valid = fact_player_game_stats['club_current_sk'].isin(dim_club['club_sk']).sum()
app_comp_valid = fact_player_game_stats['competition_sk'].isin(dim_competion['competition_sk']).sum()
app_date_valid = fact_player_game_stats['date_sk'].isin(dim_date['date_sk']).sum()

print(f"  game_sk valid refs: {app_game_valid}/{len(fact_player_game_stats)} = {app_game_valid/len(fact_player_game_stats)*100:.2f}%")
print(f"  player_sk valid refs: {app_player_valid}/{len(fact_player_game_stats)} = {app_player_valid/len(fact_player_game_stats)*100:.2f}%")
print(f"  club_sk valid refs: {app_club_valid}/{len(fact_player_game_stats)} = {app_club_valid/len(fact_player_game_stats)*100:.2f}%")
print(f"  club_current_sk valid refs: {app_club_current_valid}/{len(fact_player_game_stats)} = {app_club_current_valid/len(fact_player_game_stats)*100:.2f}%")
print(f"  competition_sk valid refs: {app_comp_valid}/{len(fact_player_game_stats)} = {app_comp_valid/len(fact_player_game_stats)*100:.2f}%")
print(f"  date_sk valid refs: {app_date_valid}/{len(fact_player_game_stats)} = {app_date_valid/len(fact_player_game_stats)*100:.2f}%")

# FACT_PLAYER_MARKET_VALUE
print("\n✓ FACT_PLAYER_MARKET_VALUE:")
print(f"  Total rows: {len(fact_player_market_value)}")
print(f"  NULL in market_value_sk: {fact_player_market_value['market_value_sk'].isna().sum()}")
print(f"  NULL in player_sk: {fact_player_market_value['player_sk'].isna().sum()}")
print(f"  NULL in date_sk: {fact_player_market_value['date_sk'].isna().sum()}")
print(f"  NULL in club_sk: {fact_player_market_value['club_sk'].isna().sum()}")

mv_player_valid = fact_player_market_value['player_sk'].isin(dim_player['player_sk']).sum()
mv_date_valid = fact_player_market_value['date_sk'].isin(dim_date['date_sk']).sum()
mv_club_valid = fact_player_market_value['club_sk'].isin(dim_club['club_sk']).sum()

print(f"  player_sk valid refs: {mv_player_valid}/{len(fact_player_market_value)} = {mv_player_valid/len(fact_player_market_value)*100:.2f}%")
print(f"  date_sk valid refs: {mv_date_valid}/{len(fact_player_market_value)} = {mv_date_valid/len(fact_player_market_value)*100:.2f}%")
print(f"  club_sk valid refs: {mv_club_valid}/{len(fact_player_market_value)} = {mv_club_valid/len(fact_player_market_value)*100:.2f}%")

# FACT_PLAYER_TRANSFER
print("\n✓ FACT_PLAYER_TRANSFER:")
print(f"  Total rows: {len(fact_player_transfer)}")
print(f"  NULL in transfer_sk: {fact_player_transfer['transfer_sk'].isna().sum()}")
print(f"  NULL in player_sk: {fact_player_transfer['player_sk'].isna().sum()}")
print(f"  NULL in transfer_date_sk: {fact_player_transfer['transfer_date_sk'].isna().sum()}")
print(f"  NULL in from_club_sk: {fact_player_transfer['from_club_sk'].isna().sum()}")
print(f"  NULL in to_club_sk: {fact_player_transfer['to_club_sk'].isna().sum()}")

tr_player_valid = fact_player_transfer['player_sk'].isin(dim_player['player_sk']).sum()
tr_date_valid = fact_player_transfer['transfer_date_sk'].isin(dim_date['date_sk']).sum()
tr_from_club_valid = fact_player_transfer['from_club_sk'].isin(dim_club['club_sk']).sum()
tr_to_club_valid = fact_player_transfer['to_club_sk'].isin(dim_club['club_sk']).sum()

print(f"  player_sk valid refs: {tr_player_valid}/{len(fact_player_transfer)} = {tr_player_valid/len(fact_player_transfer)*100:.2f}%")
print(f"  transfer_date_sk valid refs: {tr_date_valid}/{len(fact_player_transfer)} = {tr_date_valid/len(fact_player_transfer)*100:.2f}%")
print(f"  from_club_sk valid refs: {tr_from_club_valid}/{len(fact_player_transfer)} = {tr_from_club_valid/len(fact_player_transfer)*100:.2f}%")
print(f"  to_club_sk valid refs: {tr_to_club_valid}/{len(fact_player_transfer)} = {tr_to_club_valid/len(fact_player_transfer)*100:.2f}%")


✓ FACT_CLUB_GAME_STATS:
  Total rows: 10498
  NULL in club_game_sk: 0
  NULL in game_sk: 0
  NULL in club_sk: 0
  NULL in opponent_club_sk: 0
  game_sk valid refs: 10498/10498 = 100.00%
  club_sk valid refs: 10498/10498 = 100.00%
  opponent_club_sk valid refs: 10498/10498 = 100.00%

✓ FACT_PLAYER_GAME_STATS (Appearances):
  Total rows: 147550
  NULL in appearance_sk: 0
  NULL in game_sk: 0
  NULL in player_sk: 0
  NULL in club_sk: 0
  NULL in club_current_sk: 2575
  NULL in competition_sk: 0
  NULL in date_sk: 0
  game_sk valid refs: 147550/147550 = 100.00%
  player_sk valid refs: 147550/147550 = 100.00%
  club_sk valid refs: 147550/147550 = 100.00%
  club_current_sk valid refs: 144975/147550 = 98.25%
  competition_sk valid refs: 147550/147550 = 100.00%
  date_sk valid refs: 147550/147550 = 100.00%

✓ FACT_PLAYER_MARKET_VALUE:
  Total rows: 59246
  NULL in market_value_sk: 0
  NULL in player_sk: 0
  NULL in date_sk: 4642
  NULL in club_sk: 3303
  player_sk valid refs: 59246/59246 = 10

In [ ]:
print("\n" + "="*70)
print("3.3. DETAIL: ORPHANED RECORDS & INVALID REFERENCES")
print("="*70)

# FACT_GAME - orphaned records
print("\n✓ FACT_GAME Orphaned Records:")
orphaned_home = fact_game[~fact_game['home_club_sk'].isin(dim_club['club_sk'])]
orphaned_away = fact_game[~fact_game['away_club_sk'].isin(dim_club['club_sk'])]
orphaned_comp = fact_game[~fact_game['competition_sk'].isin(dim_competion['competition_sk'])]
orphaned_date = fact_game[~fact_game['date_sk'].isin(dim_date['date_sk'])]

print(f"  Home clubs not in dim_club: {len(orphaned_home)}")
if len(orphaned_home) > 0:
    print(f"    Sample: {orphaned_home[['game_sk', 'home_club_sk']].head()}")

print(f"  Away clubs not in dim_club: {len(orphaned_away)}")
if len(orphaned_away) > 0:
    print(f"    Sample: {orphaned_away[['game_sk', 'away_club_sk']].head()}")

print(f"  Competitions not in dim_competion: {len(orphaned_comp)}")
print(f"  Dates not in dim_date: {len(orphaned_date)}")

# FACT_GAME_EVENTS - orphaned records
print("\n✓ FACT_GAME_EVENTS Orphaned Records:")
orphaned_game_events_game = fact_game_events[~fact_game_events['game_sk'].isin(fact_game['game_sk'])]
orphaned_game_events_player = fact_game_events[~fact_game_events['player_sk'].isin(dim_player['player_sk'])]
orphaned_game_events_club = fact_game_events[~fact_game_events['club_sk'].isin(dim_club['club_sk'])]
orphaned_game_events_date = fact_game_events[~fact_game_events['date_sk'].isin(dim_date['date_sk'])]

print(f"  Games not in fact_game: {len(orphaned_game_events_game)}")
print(f"  Players not in dim_player: {len(orphaned_game_events_player)}")
print(f"  Clubs not in dim_club: {len(orphaned_game_events_club)}")
print(f"  Dates not in dim_date: {len(orphaned_game_events_date)}")

# FACT_PLAYER_GAME_STATS - orphaned records
print("\n✓ FACT_PLAYER_GAME_STATS Orphaned Records:")
orphaned_app_game = fact_player_game_stats[~fact_player_game_stats['game_sk'].isin(fact_game['game_sk'])]
orphaned_app_player = fact_player_game_stats[~fact_player_game_stats['player_sk'].isin(dim_player['player_sk'])]
orphaned_app_club = fact_player_game_stats[~fact_player_game_stats['club_sk'].isin(dim_club['club_sk'])]
orphaned_app_club_current = fact_player_game_stats[~fact_player_game_stats['club_current_sk'].isin(dim_club['club_sk'])]
orphaned_app_comp = fact_player_game_stats[~fact_player_game_stats['competition_sk'].isin(dim_competion['competition_sk'])]
orphaned_app_date = fact_player_game_stats[~fact_player_game_stats['date_sk'].isin(dim_date['date_sk'])]

print(f"  Games not in fact_game: {len(orphaned_app_game)}")
print(f"  Players not in dim_player: {len(orphaned_app_player)}")
print(f"  Clubs (club_sk) not in dim_club: {len(orphaned_app_club)}")
print(f"  Current clubs (club_current_sk) not in dim_club: {len(orphaned_app_club_current)}")
print(f"  Competitions not in dim_competion: {len(orphaned_app_comp)}")
print(f"  Dates not in dim_date: {len(orphaned_app_date)}")

# FACT_CLUB_GAME_STATS - orphaned records
print("\n✓ FACT_CLUB_GAME_STATS Orphaned Records:")
orphaned_cgs_game = fact_club_game_stats[~fact_club_game_stats['game_sk'].isin(fact_game['game_sk'])]
orphaned_cgs_club = fact_club_game_stats[~fact_club_game_stats['club_sk'].isin(dim_club['club_sk'])]
orphaned_cgs_opponent = fact_club_game_stats[~fact_club_game_stats['opponent_club_sk'].isin(dim_club['club_sk'])]

print(f"  Games not in fact_game: {len(orphaned_cgs_game)}")
print(f"  Clubs not in dim_club: {len(orphaned_cgs_club)}")
print(f"  Opponent clubs not in dim_club: {len(orphaned_cgs_opponent)}")

# FACT_PLAYER_TRANSFER - orphaned records
print("\n✓ FACT_PLAYER_TRANSFER Orphaned Records:")
orphaned_tr_player = fact_player_transfer[~fact_player_transfer['player_sk'].isin(dim_player['player_sk'])]
orphaned_tr_date = fact_player_transfer[~fact_player_transfer['transfer_date_sk'].isin(dim_date['date_sk'])]
orphaned_tr_from = fact_player_transfer[~fact_player_transfer['from_club_sk'].isin(dim_club['club_sk'])]
orphaned_tr_to = fact_player_transfer[~fact_player_transfer['to_club_sk'].isin(dim_club['club_sk'])]

print(f"  Players not in dim_player: {len(orphaned_tr_player)}")
print(f"  Dates not in dim_date: {len(orphaned_tr_date)}")
print(f"  From clubs not in dim_club: {len(orphaned_tr_from)}")
print(f"  To clubs not in dim_club: {len(orphaned_tr_to)}")

# FACT_PLAYER_MARKET_VALUE - orphaned records
print("\n✓ FACT_PLAYER_MARKET_VALUE Orphaned Records:")
orphaned_mv_player = fact_player_market_value[~fact_player_market_value['player_sk'].isin(dim_player['player_sk'])]
orphaned_mv_date = fact_player_market_value[~fact_player_market_value['date_sk'].isin(dim_date['date_sk'])]
orphaned_mv_club = fact_player_market_value[~fact_player_market_value['club_sk'].isin(dim_club['club_sk'])]

print(f"  Players not in dim_player: {len(orphaned_mv_player)}")
print(f"  Dates not in dim_date: {len(orphaned_mv_date)}")
print(f"  Clubs not in dim_club: {len(orphaned_mv_club)}")


3.3. DETAIL: ORPHANED RECORDS & INVALID REFERENCES

✓ FACT_GAME Orphaned Records:
  Home clubs not in dim_club: 0
  Away clubs not in dim_club: 0
  Competitions not in dim_competion: 0
  Dates not in dim_date: 0

✓ FACT_GAME_EVENTS Orphaned Records:
  Games not in fact_game: 0
  Players not in dim_player: 55505
  Clubs not in dim_club: 0
  Dates not in dim_date: 0

✓ FACT_PLAYER_GAME_STATS Orphaned Records:
  Games not in fact_game: 0
  Players not in dim_player: 0
  Clubs (club_sk) not in dim_club: 0
  Current clubs (club_current_sk) not in dim_club: 2575
  Competitions not in dim_competion: 0
  Dates not in dim_date: 0

✓ FACT_CLUB_GAME_STATS Orphaned Records:
  Games not in fact_game: 0
  Clubs not in dim_club: 0
  Opponent clubs not in dim_club: 0

✓ FACT_PLAYER_TRANSFER Orphaned Records:
  Players not in dim_player: 0
  Dates not in dim_date: 1216
  From clubs not in dim_club: 6286
  To clubs not in dim_club: 5055

✓ FACT_PLAYER_MARKET_VALUE Orphaned Records:
  Players not in dim

In [ ]:
print("\n" + "="*70)
print("3.4. SUMMARY & POTENTIAL ISSUES")
print("="*70)

# Overall quality score
total_checks = 0
passed_checks = 0

# Dim table checks
dim_checks = [
    dim_competion['competition_sk'].nunique() == len(dim_competion),
    dim_club['club_sk'].nunique() == len(dim_club),
    dim_player['player_sk'].nunique() == len(dim_player),
    dim_date['date_sk'].nunique() == len(dim_date)
]

for i, check in enumerate(dim_checks):
    total_checks += 1
    if check:
        passed_checks += 1

# Fact table FK integrity checks  
fact_checks = {
    'FACT_GAME - home_club_sk': (home_fk_valid / len(fact_game) == 1.0),
    'FACT_GAME - away_club_sk': (away_fk_valid / len(fact_game) == 1.0),
    'FACT_GAME - competition_sk': (comp_fk_valid / len(fact_game) == 1.0),
    'FACT_GAME - date_sk': (date_fk_valid / len(fact_game) == 1.0),
    'FACT_GAME_EVENTS - game_sk': (game_event_game_valid / len(fact_game_events) == 1.0 if len(fact_game_events) > 0 else True),
    'FACT_GAME_EVENTS - player_sk': (game_event_player_valid / len(fact_game_events) == 1.0 if len(fact_game_events) > 0 else True),
    'FACT_PLAYER_GAME_STATS - game_sk': (app_game_valid / len(fact_player_game_stats) == 1.0 if len(fact_player_game_stats) > 0 else True),
    'FACT_PLAYER_GAME_STATS - player_sk': (app_player_valid / len(fact_player_game_stats) == 1.0 if len(fact_player_game_stats) > 0 else True),
    'FACT_PLAYER_TRANSFER - player_sk': (tr_player_valid / len(fact_player_transfer) == 1.0 if len(fact_player_transfer) > 0 else True),
    'FACT_PLAYER_MARKET_VALUE - player_sk': (mv_player_valid / len(fact_player_market_value) == 1.0 if len(fact_player_market_value) > 0 else True),
}

for check_name, result in fact_checks.items():
    total_checks += 1
    if result:
        passed_checks += 1
        print(f"✓ {check_name}")
    else:
        print(f"✗ {check_name} - HAS ISSUES!")

print(f"\n{'─'*70}")
print(f"QUALITY SCORE: {passed_checks}/{total_checks} checks passed ({passed_checks/total_checks*100:.1f}%)")

if passed_checks == total_checks:
    print("✓✓✓ All FK mappings are VALID! Data integrity is excellent.")
else:
    print(f"⚠ {total_checks - passed_checks} issue(s) found. Please review above.")


3.4. SUMMARY & POTENTIAL ISSUES
✓ FACT_GAME - home_club_sk
✓ FACT_GAME - away_club_sk
✓ FACT_GAME - competition_sk
✓ FACT_GAME - date_sk
✓ FACT_GAME_EVENTS - game_sk
✗ FACT_GAME_EVENTS - player_sk - HAS ISSUES!
✓ FACT_PLAYER_GAME_STATS - game_sk
✓ FACT_PLAYER_GAME_STATS - player_sk
✓ FACT_PLAYER_TRANSFER - player_sk
✓ FACT_PLAYER_MARKET_VALUE - player_sk

──────────────────────────────────────────────────────────────────────
QUALITY SCORE: 13/14 checks passed (92.9%)
⚠ 1 issue(s) found. Please review above.


<h1>4. HANDLING MISSING DATA - Add Missing IDs to Dimensions</h1>

In [ ]:
print("="*70)
print("4.1. FINDING MISSING PLAYER_IDs AND CLUB_IDs")
print("="*70)

# ===== MISSING PLAYER_IDs =====
# Collect all player_ids referenced in fact tables
referenced_player_ids = set()

# From fact_game_events
if 'player_sk' in fact_game_events.columns:
    # Get player_ids from game_events
    game_events_players = uk_game_events['player_id'].dropna().unique()
    referenced_player_ids.update(game_events_players)
    
    game_events_players_in = uk_game_events['player_in_id'].dropna().unique()
    referenced_player_ids.update(game_events_players_in)
    
    game_events_players_assist = uk_game_events['player_assist_id'].dropna().unique()
    referenced_player_ids.update(game_events_players_assist)

# From appearances (player_game_stats)
app_players = uk_appearances['player_id'].dropna().unique()
referenced_player_ids.update(app_players)

# From market value
mv_players = uk_players_value['player_id'].dropna().unique()
referenced_player_ids.update(mv_players)

# From transfers
tr_players = uk_players_transfer['player_id'].dropna().unique()
referenced_player_ids.update(tr_players)

# Players in dim_player
dim_player_ids = set(dim_player['player_id'].dropna().unique())

# Find missing
missing_player_ids = referenced_player_ids - dim_player_ids

print(f"\n✓ Total referenced player_ids: {len(referenced_player_ids)}")
print(f"✓ Total in dim_player: {len(dim_player_ids)}")
print(f"✗ Missing player_ids: {len(missing_player_ids)}")

if len(missing_player_ids) > 0:
    print(f"\n  Sample missing player_ids: {sorted(list(missing_player_ids))[:10]}")
    
    # Lấy info về missing players từ original players table
    missing_players_info = players[players['player_id'].isin(missing_player_ids)]
    print(f"  Found info for {len(missing_players_info)} missing players in raw data")

4.1. FINDING MISSING PLAYER_IDs AND CLUB_IDs

✓ Total referenced player_ids: 2486
✓ Total in dim_player: 2473
✗ Missing player_ids: 13

  Sample missing player_ids: [11756, 36894, 120599, 570336.0, 914269.0, 926309.0, 933018, 999743.0, 1023191.0, 1047280.0]
  Found info for 0 missing players in raw data


In [ ]:
print("\n" + "="*70)
print("4.2. ADDING MISSING PLAYERS TO DIM_PLAYER")
print("="*70)

if len(missing_player_ids) > 0:
    print(f"\nAdding {len(missing_player_ids)} missing players...")
    
    # Lấy info từ raw players table
    missing_players_to_add = players[players['player_id'].isin(missing_player_ids)].copy()
    
    # Select only columns that exist in dim_player
    dim_player_cols = dim_player.columns.drop('player_sk')
    cols_available = [col for col in dim_player_cols if col in missing_players_to_add.columns]
    
    # Create new rows
    new_players = missing_players_to_add[cols_available].copy()
    
    # Fill missing values with same logic as before
    for col in new_players.columns:
        if new_players[col].isna().any():
            if col == 'foot':
                mask = new_players[col].isna()
                new_players.loc[mask, col] = np.random.choice(['left', 'right', 'both'], size=mask.sum())
            elif col == 'height_in_cm':
                mask = new_players[col].isna()
                new_players.loc[mask, col] = np.random.randint(175, 200, size=mask.sum())
            elif col == 'date_of_birth':
                mask = new_players[col].isna()
                new_players.loc[mask, col] = pd.to_datetime(
                    2026 - np.random.randint(18, 41, size=mask.sum()), format='%Y'
                ) + pd.to_timedelta(np.random.randint(0, 365, size=mask.sum()), unit='D')
            elif col in ['market_value_in_eur', 'highest_market_value_in_eur']:
                mask = new_players[col].isna()
                values = np.random.lognormal(mean=np.log(5_000_000), sigma=1.0, size=mask.sum())
                values = np.clip(values, 100_000, 150_000_000)
                new_players.loc[mask, col] = values.round()
            else:
                # Fillna with appropriate defaults
                if new_players[col].dtype in ['float64', 'int64']:
                    new_players[col] = new_players[col].fillna(new_players[col].median())
                else:
                    new_players[col] = new_players[col].fillna('Unknown')
    
    # Add player_sk  
    max_player_sk = dim_player['player_sk'].max()
    new_players['player_sk'] = range(max_player_sk + 1, max_player_sk + len(new_players) + 1)
    
    # Reorder columns to match dim_player
    new_players = new_players[dim_player.columns]
    
    # Append to dim_player
    dim_player = pd.concat([dim_player, new_players], ignore_index=True)
    
    print(f"✓ Added {len(new_players)} players to dim_player")
    print(f"✓ dim_player new size: {len(dim_player)}")
    print(f"✓ New player_sk range: {new_players['player_sk'].min()} - {new_players['player_sk'].max()}")
else:
    print("✓ No missing player_ids found!")


4.2. ADDING MISSING PLAYERS TO DIM_PLAYER

Adding 13 missing players...
✓ Added 0 players to dim_player
✓ dim_player new size: 2473
✓ New player_sk range: nan - nan
